In [ ]:
import torch
from torch.utils.data import DataLoader
import os
import json
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import GroupKFold


from src.dataset import PatchGridDataset, PatchDataset
from src.histology_feature_extractor import extract_features
from src.train_test import train_lp, test_lp, test_lp_MIL
from utils.metrics import get_metrics
from utils.set_seed import seed_everything
from utils.load_model import load_histo_model

seed = 123
seed_everything(seed)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

## 1. Load pre-trained model and transformer

In [2]:
pretrained_model_name = ['uni', 'musk', 'phikon-v2', 'plip'][2]
pre_model, transform = load_histo_model(pretrained_model_name, device, ckpts_dir='./assets/ckpts')

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


## 2. Prepare Dataset

### 2.1. Load data

In [3]:
dataset_path = "/path/to/UPMC_HE"  # TODO: path to UPMC_HE image directory
label_csv = "/path/to/upmc_labels.csv"  # TODO: path to upmc_labels.csv
patient_info_csv = "/path/to/region_id_mapping.csv"  # TODO: path to region_id_mapping.csv
label_name = 'primary_outcome'
PATCH_R = 200
PATCH_SIZE = PATCH_R * 2

label_df = pd.read_csv(label_csv)[['region_id', label_name]]
patient_info_df = pd.read_csv(patient_info_csv)[['ACQUISITION_ID', 'PATIENT_ID']]
df_merge = pd.merge(label_df, patient_info_df, left_on="region_id", right_on="ACQUISITION_ID", how="inner")
df_merge = df_merge.drop(columns=["ACQUISITION_ID"]).set_index("region_id", inplace=False)

data_names = []
for name in os.listdir(dataset_path):
    if name in df_merge.index:
        if os.path.isdir(os.path.join(dataset_path, name, "he_img")):
            data_names.append(name)
data_names = np.array(data_names)

print(f'Sample amount: {len(data_names)}')

Sample amount: 283


### 2.2. Group K-Fold Data Split (by Patient)

In [ ]:
# # Order regions to ensure consistent split
# data_names = np.sort(data_names)
# np.random.shuffle(data_names)

# n_splits = 4
# patient_ids = [df_merge.loc[name, 'PATIENT_ID'] for name in data_names]
# groups = np.array(patient_ids)  # Patient groups
# split_indices = {
#     fold: (data_names[train_idx].tolist(), data_names[test_idx].tolist()) 
#     for fold, (train_idx, test_idx) in
#     enumerate(GroupKFold(n_splits=n_splits).split(data_names, groups=groups))}


with open('./data/upmc_split_indices.json', 'r') as f:
    split_indices = json.load(f)

## 3. Train and test

In [5]:
print(f'Foundation model is {pretrained_model_name}.')

# Cross validation
linprobe_test_results = []
for fold, (train_idx, test_idx) in split_indices.items():
    seed_everything(seed)
    print(f"----------")

    anno_csv = f"upmc_microE_anno_primary_outcome_patientGrp{fold}.csv"
    anno_df = pd.read_csv(os.path.join('./data/microE_annotations', anno_csv)).set_index('region_id', inplace=False)

    # Ensure no data leakage: microE annotations constructed exclusively from training set
    assert set(train_idx) == set(anno_df.index)

    train_df_merge = df_merge.loc[train_idx]
    train_df_merge = pd.merge(train_df_merge, anno_df, left_index=True, right_index=True, how="inner")
    print(f"Fold {fold}: There are {len(train_df_merge)} patches for training.")

    # Training dataset
    zipped_crop_ranges = list(zip(train_df_merge['centerX'].values - PATCH_R, train_df_merge['centerY'].values - PATCH_R,\
                                train_df_merge['centerX'].values + PATCH_R, train_df_merge['centerY'].values + PATCH_R))
    train_crop_ranges = np.array([list(item) for item in zipped_crop_ranges])

    train_dataset = PatchDataset(img_names=train_df_merge.index.to_numpy(), 
                                target_labels=train_df_merge[label_name].to_numpy(),
                                aux_labels=[],  # Baseline: without microE annotations
                                transform=transform, 
                                dataset_dir=dataset_path,
                                is_crop=True, 
                                crop_range=train_crop_ranges,
                                return_region=False)
    train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=8)
    train_features = extract_features(pre_model, pretrained_model_name, train_dataloader, device=device)
    train_feats = train_features['embeddings']
    train_labels = train_features['labels']
    
    # Train
    classifier = train_lp(train_feats, train_labels, device, epochs=1)
  

    # Test
    test_labels_true = []
    test_labels_pred = []
    test_labels_prob = []

    test_df_merge = df_merge.loc[test_idx]
    for test_name, test_info in tqdm(test_df_merge.iterrows(), total=len(test_df_merge), desc='Testing'):
        test_label = test_info[label_name]
        test_dataset = PatchGridDataset(img_names=[test_name], 
                                        patch_size=PATCH_SIZE, 
                                        stride=PATCH_SIZE, 
                                        target_labels=[test_label],
                                        transform=transform, 
                                        dataset_dir=dataset_path)
        test_dataloader = DataLoader(test_dataset, batch_size=16, shuffle=True, num_workers=8)
        test_features = extract_features(pre_model, pretrained_model_name, test_dataloader, device=device, show_progress=False)
        test_feats = test_features['embeddings']
        test_labels = test_features['labels']

        test_pred, test_prob = test_lp_MIL(test_feats, classifier, device)
        
        test_labels_true.append(test_label)
        test_labels_prob.append(test_prob)
        test_labels_pred.append(test_pred)

    classifier.to('cpu')
    del classifier
    torch.cuda.empty_cache()

    linprobe_eval_result = get_metrics(test_labels_true, test_labels_pred, test_labels_prob, get_report=True)
    linprobe_test_results.append(linprobe_eval_result)
    print(f"Fold{fold}: auroc = {linprobe_eval_result['auroc']:.3f}")


print(f"\nAverage results: auroc = {pd.DataFrame(linprobe_test_results)['auroc'].mean():.3f}")

Foundation model is phikon-v2.
----------
Fold 0: There are 32767 patches for training.


extracting features: 100%|██████████| 2048/2048 [09:25<00:00,  3.62it/s]


Train Epoch: 1 	 Loss: 0.443499


Testing: 100%|██████████| 71/71 [02:05<00:00,  1.77s/it]


Fold0: auroc = 0.854
----------
Fold 1: There are 32766 patches for training.


...

Fold2: auroc = 0.659
----------
Fold 3: There are 32767 patches for training.


extracting features: 100%|██████████| 2048/2048 [08:52<00:00,  3.85it/s]


Train Epoch: 1 	 Loss: 0.392463


Testing: 100%|██████████| 70/70 [01:37<00:00,  1.40s/it]

Fold3: auroc = 0.750

Average results: auroc = 0.726
